# 02 - Classification Data (Stage 2)
Pregateste datele pentru clasificare: stats, split pentru park crops (daca exista), merge cu TrashNet in mixed_cls.

In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_ROOT = Path('../..').resolve()
py = sys.executable

def run(cmd):
    print('\n>', ' '.join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=REPO_ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed: {completed.returncode}')

In [ ]:
PARKS_POOL = REPO_ROOT / 'datasets' / 'parks_cls_pool'
PARKS_CLS = REPO_ROOT / 'datasets' / 'parks_cls'
TRASHNET_CLS = REPO_ROOT / 'datasets' / 'trashnet_cls'
MIXED_CLS = REPO_ROOT / 'datasets' / 'mixed_cls'
MANIFEST = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'crops_manifest.csv'

print('parks_cls_pool exists:', PARKS_POOL.exists())
print('parks_cls exists     :', PARKS_CLS.exists())
print('trashnet_cls exists  :', TRASHNET_CLS.exists())

In [ ]:
# Ruleaza doar daca ai sortat manual crop-urile in datasets/parks_cls_pool/<class>/
if PARKS_POOL.exists():
    run([
        py, 'scripts/split_classification_dataset.py',
        '--source-root', str(PARKS_POOL),
        '--out-root', str(PARKS_CLS),
        '--manifest', str(MANIFEST),
        '--group-by', 'source-image',
        '--val', '0.15', '--test', '0.15', '--seed', '42', '--clear'
    ])
else:
    print('Skip split_classification_dataset: parks_cls_pool nu exista inca.')

In [ ]:
if TRASHNET_CLS.exists() and PARKS_CLS.exists():
    run([
        py, 'scripts/merge_classification_datasets.py',
        '--datasets', str(TRASHNET_CLS), str(PARKS_CLS),
        '--out-dir', str(MIXED_CLS)
    ])
else:
    print('Skip merge: trebuie sa existe trashnet_cls si parks_cls.')

In [ ]:
for ds in [TRASHNET_CLS, PARKS_CLS, MIXED_CLS]:
    if ds.exists():
        run([py, 'scripts/report_classification_dataset_stats.py', '--data', str(ds)])
    else:
        print('Missing dataset:', ds)

# 02 — Pregătire Dataset Clasificare

Acest notebook acoperă **Stage 2 (clasificare materiale)** — pregătire date:

1. Split dataset clasificare (crops din parks)
2. Statistici dataset clasificare
3. Merge dataset-uri clasificare (TrashNet + parks crops → mixed_cls)

**Pre-condiție**: Ai rulat **01_data_preparation.ipynb** și ai crops în `datasets/parks_cls_unsorted/all`.

Clasele: `glass | metal | other | paper | plastic`

In [ ]:
import random
import shutil
import sys
import csv
from collections import defaultdict
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASSES    = ["glass", "metal", "other", "paper", "plastic"]

print(f"Rădăcina proiectului: {REPO_ROOT}")

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

# Split parks crops
CLS_SOURCE_ROOT  = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "all"
CLS_OUT_ROOT     = REPO_ROOT / "datasets" / "parks_cls"
CLS_MANIFEST     = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "crops_manifest.csv"
CLS_VAL_RATIO    = 0.15
CLS_TEST_RATIO   = 0.15
CLS_SEED         = 42
CLS_GROUP_BY     = "source-image"   # "source-image" sau "crop"
CLS_COPY         = True
CLS_CLEAR        = True

# Merge clasificare
TRASHNET_DIR     = REPO_ROOT / "datasets" / "trashnet_cls"
MIXED_OUT_DIR    = REPO_ROOT / "datasets" / "mixed_cls"

print("Config OK")

---
## 1. Split Dataset Clasificare

Împarte crops-urile adnotate din `parks_cls_unsorted` în train/val/test.  
Gruparea pe `source-image` previne leakage-ul — crops din aceeași imagine sursă nu vor fi în split-uri diferite.

> **Notă**: Structura așteptată a sursei este `parks_cls_unsorted/all/<clasa>/imagine.jpg`  
> Dacă ai crops nesortate și le-ai sortat manual pe clase, rulează această celulă.

In [ ]:
def iter_images(directory: Path):
    if not directory.exists(): return []
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def load_manifest(manifest_path: Path):
    if not manifest_path.exists(): return {}
    mapping = {}
    with manifest_path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            mapping[Path(row["crop_path"]).name] = Path(row["source_image"]).stem
    return mapping


def assign_groups(group_keys, val_ratio, test_ratio, seed):
    shuffled = list(group_keys)
    random.Random(seed).shuffle(shuffled)
    total = len(shuffled)
    tc = int(round(total * test_ratio)); vc = int(round(total * val_ratio))
    if total >= 3:
        if test_ratio > 0 and tc == 0: tc = 1
        if val_ratio  > 0 and vc == 0: vc = 1
        ov = tc + vc - total + 1
        while ov > 0 and tc > 0: tc -= 1; ov -= 1
        while ov > 0 and vc > 0: vc -= 1; ov -= 1
    sm = {}
    for i, k in enumerate(shuffled):
        sm[k] = "test" if i < tc else ("val" if i < tc + vc else "train")
    return sm


def split_classification_dataset(
    source_root, out_root, manifest_path, classes,
    val_ratio, test_ratio, seed, group_by, copy_files, clear
):
    source_root = Path(source_root); out_root = Path(out_root)
    if not source_root.exists():
        print(f"[EROARE] Sursă inexistentă: {source_root}"); return

    manifest_map = load_manifest(Path(manifest_path)) if group_by == "source-image" else {}

    grouped_items = defaultdict(list)
    for cls in classes:
        cls_dir = source_root / cls
        if not cls_dir.exists(): continue
        for img in iter_images(cls_dir):
            key = manifest_map.get(img.name, img.stem) if group_by == "source-image" else img.stem
            grouped_items[(cls, key)].append(img)

    if not grouped_items:
        print(f"[EROARE] Nicio imagine în {source_root}"); return

    if clear:
        for split in ("train", "val", "test"):
            d = out_root / split
            if d.exists(): shutil.rmtree(d)

    for split in ("train", "val", "test"):
        for cls in classes:
            (out_root / split / cls).mkdir(parents=True, exist_ok=True)

    grouped_by_class = defaultdict(list)
    for (cls, key) in grouped_items:
        grouped_by_class[cls].append(key)

    stats = {sp: defaultdict(int) for sp in ("train","val","test")}
    for cls in classes:
        keys = grouped_by_class.get(cls, [])
        if not keys: continue
        sm = assign_groups(keys, val_ratio, test_ratio, seed)
        for key in keys:
            sp = sm[key]
            for src in grouped_items[(cls, key)]:
                dst = out_root / sp / cls / src.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                if copy_files: shutil.copy2(src, dst)
                else: shutil.move(str(src), str(dst))
                stats[sp][cls] += 1

    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)
    for cls in classes:
        tr = stats['train'][cls]; va = stats['val'][cls]; te = stats['test'][cls]
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")
    print("\nSplit clasificare generat!")


split_classification_dataset(
    CLS_SOURCE_ROOT, CLS_OUT_ROOT, CLS_MANIFEST, CLASSES,
    CLS_VAL_RATIO, CLS_TEST_RATIO, CLS_SEED, CLS_GROUP_BY, CLS_COPY, CLS_CLEAR
)

---
## 2. Statistici Dataset Clasificare
Afișează distribuția imaginilor pe clase și split-uri.

In [ ]:
def report_classification_stats(data_root: Path, classes):
    if not data_root.exists():
        print(f"[EROARE] Nu există: {data_root}"); return

    def count_imgs(d): return sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if d.exists() else 0

    print(f"Dataset: {data_root.name}")
    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)

    grand = {"train": 0, "val": 0, "test": 0}
    for cls in classes:
        tr = count_imgs(data_root / "train" / cls)
        va = count_imgs(data_root / "val"   / cls)
        te = count_imgs(data_root / "test"  / cls)
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")
        grand["train"] += tr; grand["val"] += va; grand["test"] += te

    print("-" * 45)
    print(f"{'TOTAL':<12} {grand['train']:>7} {grand['val']:>7} {grand['test']:>7} {sum(grand.values()):>8}")


print("=== parks_cls ===")
report_classification_stats(REPO_ROOT / "datasets" / "parks_cls", CLASSES)

In [ ]:
print("=== trashnet_cls ===")
report_classification_stats(TRASHNET_DIR, CLASSES)
print()
print("=== mixed_cls ===")
report_classification_stats(MIXED_OUT_DIR, CLASSES)

---
## 3. Merge Dataset-uri Clasificare
Combină **TrashNet** + **parks_cls** în `mixed_cls` (folosit pentru Experimentul B3 din experiment plan).  

**Pre-condiție**: Ai rulat `scripts/download_trashnet.py` pentru a descărca TrashNet.

In [ ]:
def merge_classification_datasets(sources: list, out_dir: Path, classes):
    sources = [Path(s) for s in sources]
    for src in sources:
        if not src.exists():
            print(f"[WARN] Sursă inexistentă: {src}")

    if out_dir.exists():
        print(f"Șterg output existent: {out_dir}")
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    totals = defaultdict(lambda: defaultdict(int))

    for src in sources:
        if not src.exists(): continue
        for split in ("train", "val", "test"):
            for cls in classes:
                src_cls = src / split / cls
                if not src_cls.exists(): continue
                dst_cls = out_dir / split / cls
                dst_cls.mkdir(parents=True, exist_ok=True)
                imgs = list(src_cls.glob("*.jpg")) + list(src_cls.glob("*.png")) + list(src_cls.glob("*.jpeg"))
                for img in imgs:
                    dst = dst_cls / f"{src.name}_{img.name}"
                    shutil.copy2(img, dst)
                    totals[cls][split] += 1
        print(f"Procesat: {src.name}")

    print(f"\nDataset merged: {out_dir}")
    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)
    for cls in classes:
        tr = totals[cls]["train"]; va = totals[cls]["val"]; te = totals[cls]["test"]
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")


merge_classification_datasets(
    [TRASHNET_DIR, REPO_ROOT / "datasets" / "parks_cls"],
    MIXED_OUT_DIR,
    CLASSES
)